In [4]:
import os
import time
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import autocast, GradScaler
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup_distributed():
    dist.init_process_group(backend='nccl', init_method='env:://'

# デバイスの設定
def setup_distributed():
    dist.init_process_group(backend='nccl')
    torch.cuda.set_device(int(os.environ['LOCAL_RANK']))
    device = torch.device('cuda', int(os.environ['LOCAL_RANK']))
    return device

device = setup_distributed()
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# 座標データのロード（最大3654行）
coords = ["x", "y", "z"]
coords_data = {coord: np.load(f"/home/nishioka/GNN/BasicdataforGNN/{coord}_2layer_normalized.npy")[:3654] for coord in coords}

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# 欠陷なしデータのペア
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")

# 初期のペアリスト
train_pairs = []
val_pairs = []
test_pairs = []

# 欠陷なしデータを追加
for _ in range(8):
    train_pairs.append(defect_free_pair)
val_pairs.append(defect_free_pair)
test_pairs.append(defect_free_pair)

# ペアのデータをカウントして確認
def count_pair_occurrences(pairs, target_pair):
    return pairs.count(target_pair)

train_count = count_pair_occurrences(train_pairs, defect_free_pair)
val_count = count_pair_occurrences(val_pairs, defect_free_pair)
test_count = count_pair_occurrences(test_pairs, defect_free_pair)

print(f"\nDefect-free data occurrences in train_pairs: {train_count} (expected: 8)")
print(f"Defect-free data occurrences in val_pairs: {val_count} (expected: 1)")
print(f"Defect-free data occurrences in test_pairs: {test_count} (expected: 1)")

if train_count == 8 and val_count == 1 and test_count == 1:
    print("\nAll defect-free data pairs are correctly added.")
else:
    print("\nThere is an issue with adding defect-free data pairs.")

# データとラベルのペア作成
def extract_layer_block(file_name):
    """ファイル名から層とブロック番号を抽出
    """
    if file_name.startswith("0"):
        return (0, 0)
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer = int(layer_block_str.split("L")[1].split("B")[0])
        block = int(layer_block_str.split("B")[1])
        return (layer, block)
    except (ValueError, IndexError):
        print(f"Invalid file name format: {file_name}")
        return None

data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアのみ取得
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]

# ランダムにサンプリング
num_samples = 1296
random_indices = np.random.choice(len(valid_pairs), num_samples, replace=False)

# サンプルをトレーニングデータとして取得
train_pairs = [valid_pairs[i] for i in random_indices]

# 残りのデータを取得
remaining_pairs = [valid_pairs[i] for i in range(len(valid_pairs)) if i not in random_indices]

# 残りのデータを1:1でバリデーションとテストに分割
val_pairs, test_pairs = train_test_split(remaining_pairs, test_size=0.5, random_state=42)

# データ準備関数
def prepare_data(pairs):
    """データとラベルを準備し、テンソルに変換
    """
    sampled_data, sampled_labels = [], []
    for data_file, label_file in pairs:
        data_path = os.path.join(standardized_data_folder, data_file)
        label_path = os.path.join(label_data_folder, label_file)

        values = np.load(data_path)[:3654]
        label = np.load(label_path)[:3654]

        node_features = np.vstack((coords_data["x"], coords_data["y"], coords_data["z"], values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)

    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

# 各データセットを準備
train_x, train_y = prepare_data(train_pairs)
val_x, val_y = prepare_data(val_pairs)
test_x, test_y = prepare_data(test_pairs)

# Dataオブジェクト作成
train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
val_data = Data(x=val_x, edge_index=edge_index, y=val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# モデル定義
class GATModel(torch.nn.Module):
    def __init__(self, hidden_channels=128):  # hidden_channels を 128 に増やす
        super(GATModel, self).__init__()
        self.conv1 = GATConv(4, hidden_channels, heads=8)  # heads を 8 に増やす
        self.conv2 = GATConv(hidden_channels * 8, hidden_channels * 4, heads=4)  # heads を 4 に増やす
        self.conv3 = GATConv(hidden_channels * 4, hidden_channels * 2, heads=2)
        self.fc_defect = torch.nn.Linear(hidden_channels * 2, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = self.fc_defect(x)
        return torch.sigmoid(x)

# モデル、損失関数、オプティマイザの設定
model = GATModel(hidden_channels=128).to(device)
model = DDP(model, device_ids=[int(os.environ['LOCAL_RANK'])])
model.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if isinstance(m, torch.nn.Linear) else None)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
loss_fn = torch.nn.MSELoss()

# TensorBoard設定
if int(os.environ['LOCAL_RANK']) == 0:
    writer = SummaryWriter(log_dir=f'/home/nishioka/GNN/GNNlogs/{type(model).__name__}_{timestamp}')

# データローダーの作成
train_loader = DataLoader([train_data], batch_size=32, shuffle=True)  # バッチサイズを 32 に増やす
val_loader = DataLoader([val_data], batch_size=32)
test_loader = DataLoader([test_data], batch_size=32)

# Early Stopping定義
class EarlyStopping:
    def __init__(self, patience=500, path='/home/nishioka/GNN/{type(model).__name__}_checkpoint.pt'):
        self.patience = patience
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.path = path
        self.counter = 0

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 学習
early_stopping = EarlyStopping()
train_losses, val_losses = [], []
scaler = GradScaler()

start_time = time.time()

for epoch in range(1, 2001):
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        # 混合精度を使用した順伝操作
        with autocast():
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1))
        # 減少スケーラーを使用
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    train_losses.append(train_loss / len(train_loader))

    # 検証
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1))
            val_loss += loss.item()
    val_losses.append(val_loss / len(val_loader))

    early_stopping(val_loss / len(val_loader), model)
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

    if epoch % 50 == 0:
        print(f'Epoch {epoch}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}')

# テストデータで評価
model.load_state_dict(torch.load(early_stopping.path))
model.eval()
test_loss, test_preds, test_targets = 0, [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        test_loss += loss.item()
        test_preds.append(out.cpu().numpy())
        test_targets.append(batch.y.cpu().numpy())

# 評価指標の計算
test_preds, test_targets = np.concatenate(test_preds), np.concatenate(test_targets)
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))
mae = mean_absolute_error(test_targets, test_preds)
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# 結果の保存やプロット
if int(os.environ['LOCAL_RANK']) == 0:
    elapsed_time = time.time() - start_time

    # モデル、学習結果の保存
    torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gcn_model_final_{timestamp}.pth')

    # 学習時間をファイルに保存
    with open(f'/home/nishioka/GNN/GNNmodelcsv/training_time_{timestamp}.txt', 'w') as f:
        f.write(f"Total training time: {elapsed_time:.2f} seconds\n")

    # 損失とその他の指標をCSVに保存
    loss_data = pd.DataFrame({
        'Epoch': range(1, len(train_losses) + 1),
        'Training Loss': train_losses,
        'Validation Loss': val_losses
    })
    loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{timestamp}.csv', index=False)

    # 損失をプロット
    plt.figure(figsize=(12, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', color='blue')
    plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='orange')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (Log Scale)')
    plt.title(f'Training and Validation Loss {type(model).__name__} - ({timestamp})')
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/loss_fig_{type(model).__name__}_{timestamp}.png')
    plt.show()

    test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)
    
    # モデルの保存（ファイル名にタイムスタンプを追加）
    torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_{test_loss:.4f}_{timestamp}.pth')
    torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodelweights/{type(model).__name__}_weights_{test_loss:.4f}_{timestamp}.pth')
    
    # TensorBoardのWriterを閉じる
    writer.close()
    
    # Improved Final summary after the training loop
    # def print_summary(epoch, early_stop):
    print("\nTraining Summary:")
    print(f"Total Epochs Run: {epoch}")
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
    else:
        print("Training continued without early stopping.")
    print(f"Best Epoch (Lowest Validation Loss): {epoch}")
    print(f"Final Training Loss: {train_losses[-1]:.4f}")
    print(f"Best Validation Loss: {best_val_loss:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"R2 Score: {r2:.4f}")
    print(f"Final Learning Rate: {learning_rate}")
    print(f"Training Samples: {num_train_samples}")
    print(f"Validation Samples: {num_val_samples}")
    print(f"Test Samples: {num_test_samples}")
    print(f"Total Training Time: {elapsed_time:.2f} seconds")

ValueError: Error initializing torch.distributed using env:// rendezvous: environment variable RANK expected, but not set